In [1]:
import pandas as pd
import numpy as np

In [26]:
df = pd.DataFrame({'Col_1': [2,6,7,2,9,3,6,2,7,2,6,2,3,7,9,5,6,4,2,8,18,4,7,2,6,-12,6,7,3,8],
                   'Col_2':np.random.randint(1, 10, size=30),
                   'Col_3':np.random.randint(1, 50, size=30)})


In [27]:
numeric = df.select_dtypes(include='number').columns
numeric

Index(['Col_1', 'Col_2', 'Col_3'], dtype='object')

In [28]:
iqr = {col: {'q1': np.quantile(df[col], q=0.25),
             'q3': np.quantile(df[col], q=0.75),
             'iqr' : np.quantile(df[col], q=0.75) - np.quantile(df[col], q=0.25)} for col in numeric}
iqr

{'Col_1': {'q1': 2.25, 'q3': 7.0, 'iqr': 4.75},
 'Col_2': {'q1': 3.0, 'q3': 8.0, 'iqr': 5.0},
 'Col_3': {'q1': 8.25, 'q3': 32.75, 'iqr': 24.5}}

In [29]:
bounds = {col : {'lower_bound' : iqr[col]['q1'] - (iqr[col]['iqr'] * 1.5),
                 'upper_bound' : iqr[col]['q3'] + (iqr[col]['iqr'] * 1.5)}
                 for col in numeric}
bounds

{'Col_1': {'lower_bound': -4.875, 'upper_bound': 14.125},
 'Col_2': {'lower_bound': -4.5, 'upper_bound': 15.5},
 'Col_3': {'lower_bound': -28.5, 'upper_bound': 69.5}}

In [36]:
df[df['Col_1'] < bounds['Col_1']['lower_bound']].value_counts().sum() + df[df['Col_1'] > bounds['Col_1']['upper_bound']].value_counts().sum()

2

In [37]:
outliers = {col : (df[df[col]>bounds[col]['upper_bound']].value_counts().sum() + df[df[col]<bounds[col]['lower_bound']].value_counts().sum()) for col in numeric}
outliers

{'Col_1': 2, 'Col_2': 0, 'Col_3': 0}

In [44]:
o_per = {col :round((outliers[col] / len(df[col])) * 100, 2) for col in numeric}
max(o_per.values())

6.67

In [40]:
cols = [col for col in numeric if outliers[col]>0]
cols

['Col_1']